# Do e-values help reduce the sample size of CTs?

In [1]:
# Expected sample size: calibrated Bayes vs e-value, sequential monitoring
#
# Model: outcomes x_i ~ N(true_eff, 1), checked after every patient up to nmax.
# Prior on the effect: N(guess, guess_sd^2). Same prior feeds both rules.
#
# Rules
#   e-value: mixture likelihood ratio (= Bayes factor under the prior) >= 1/alpha.
#            Valid at any stopping time by Ville's inequality, no calibration needed.
#   bayes (criterion 1, calibrated): P(effect > 0 | data) >= c, where c is chosen
#            by simulation so that under true_eff = 0 the rule fires by nmax with
#            probability alpha.
#
# Output: expected N, where a trial that never fires counts as nmax (so it is
# E[min(N, nmax)]), plus the share of trials that never fired.

alpha    <- 0.025          # one-sided, FDA convention
nmax     <- 2000
reps     <- 2000           # trials per cell
reps_cal <- 20000          # null trials per prior for calibration
guess_sd <- 0.3            # prior worth ~ 1/guess_sd^2 = 11 patients
guesses  <- c(0, 0.1, 0.2, 0.3, 0.5)
true_eff <- c(0, 0.1, 0.2, 0.3, 0.5)

# running totals S[r, n] for `reps` trials of length nmax
paths <- function(eff, reps, nmax) {
  X <- matrix(rnorm(reps * nmax, eff, 1), reps, nmax)
  for (j in 2:nmax) X[, j] <- X[, j - 1] + X[, j]   # column cumsum, fast
  X
}

# log e-value at every (trial, n)
log_e <- function(S, guess, guess_sd) {
  n <- col(S)
  dnorm(S, n * guess, sqrt(n + n^2 * guess_sd^2), log = TRUE) -
    dnorm(S, 0, sqrt(n), log = TRUE)
}

# posterior P(effect > 0) at every (trial, n)
p_pos <- function(S, guess, guess_sd) {
  prec <- 1 / guess_sd^2 + col(S)
  pnorm((guess / guess_sd^2 + S) / sqrt(prec))
}

# first column where M is TRUE, NA if never
first_hit <- function(M) {
  h <- max.col(M, ties.method = "first")
  h[!M[cbind(seq_len(nrow(M)), h)]] <- NA
  h
}

# FDA criterion 1: bar c such that P(max_n posterior >= c | effect = 0) = alpha
calibrate <- function(guess, guess_sd) {
  S <- paths(0, reps_cal, nmax)
  # work on the z scale to avoid pnorm rounding to 1
  z <- (guess / guess_sd^2 + S) / sqrt(1 / guess_sd^2 + col(S))
  zmax <- apply(z, 1, max)
  pnorm(quantile(zmax, 1 - alpha, names = FALSE))
}

summ <- function(N) c(EN = mean(ifelse(is.na(N), nmax, N)),
                      never = mean(is.na(N)))

set.seed(1)
out <- list()
for (g in guesses) {
  bar <- calibrate(g, guess_sd)
  for (t in true_eff) {
    S  <- paths(t, reps, nmax)     # same paths for both rules
    Ne <- first_hit(log_e(S, g, guess_sd) >= log(1 / alpha))
    Nb <- first_hit(p_pos(S, g, guess_sd) >= bar)
    se <- summ(Ne); sb <- summ(Nb)
    out[[length(out) + 1]] <- data.frame(
      guess = g, true_eff = t, bayes_bar = bar,
      EN_bayes = sb[["EN"]], never_bayes = sb[["never"]],
      EN_e = se[["EN"]], never_e = se[["never"]],
      ratio_e_over_bayes = se[["EN"]] / sb[["EN"]])
  }
}
res <- do.call(rbind, out)
print(format(res, digits = 3), row.names = FALSE)

# wide table: expected N under criterion 1, rows = true effect, cols = prior guess
wide <- xtabs(EN_bayes ~ true_eff + guess, res)
cat("\nE[N], calibrated Bayes (criterion 1)\n"); print(round(wide))
cat("\nE[N], e-value\n"); print(round(xtabs(EN_e ~ true_eff + guess, res)))

 guess true_eff bayes_bar EN_bayes never_bayes   EN_e never_e
   0.0      0.0     0.998   1963.9      0.9755 1971.7   0.983
   0.0      0.1     0.998    735.1      0.0280 1009.8   0.109
   0.0      0.2     0.998    207.1      0.0000  251.0   0.000
   0.0      0.3     0.998    101.4      0.0000  114.2   0.000
   0.0      0.5     0.998     43.9      0.0000   45.8   0.000
   0.1      0.0     0.998   1956.2      0.9720 1971.1   0.984
   0.1      0.1     0.998    757.1      0.0285 1024.1   0.122
   0.1      0.2     0.998    205.5      0.0000  245.5   0.000
   0.1      0.3     0.998     97.0      0.0000  107.7   0.000
   0.1      0.5     0.998     41.1      0.0000   42.0   0.000
   0.2      0.0     0.999   1958.0      0.9740 1966.7   0.982
   0.2      0.1     0.999    789.1      0.0400 1018.2   0.121
   0.2      0.2     0.999    200.6      0.0000  232.0   0.000
   0.2      0.3     0.999     93.8      0.0000  101.0   0.000
   0.2      0.5     0.999     40.1      0.0000   39.9   0.000
   0.3  